In [0]:
from pyspark.sql import functions as F

silver = spark.table("workspace.silver.gho_clean")
required = ["country_code", "region", "year", "indicator_code", "numeric_value"]

In [0]:
n_duplicates = (silver.groupBy("country_code", "year", "indicator_code")
    .count().filter("count > 1").count())
assert n_duplicates == 0, f"{n_duplicates} duplicated keys"

In [0]:
n_indicators = silver.select("indicator_code").distinct().count()
assert n_indicators == 15, f"{n_indicators} indicators, expected 15"

In [0]:
dtype = dict(silver.dtypes)["numeric_value"]
assert dtype == "double", f"numeric_value is {dtype}, expected double"

In [0]:
row = silver.select([
    F.avg((F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == "")).cast("int")).alias(c)
    for c in required
]).first()
assert row is not None, "silver is empty"
failures = {c: round(rate, 4) for c, rate in row.asDict().items() if rate > 0}
assert not failures, f"Nulls/empty strings in required columns: {failures}"

In [0]:
silver.printSchema()